# 深層学習の一気通貫（CNN）

実践編（総合演習） / End-to-end deep learning with a CNN

小さな画像データセット（scikit-learn 付属の手描き数字 `load_digits`、8x8ピクセル）を題材に、**前処理 → CNNの定義 → 学習 → 評価**までを一気通貫でたどります。前処理・深層学習・評価という単体で学んだ技術を、1本の流れにつなぐ総合演習です。

- データ: `sklearn.datasets.load_digits`（1797枚・8x8・10クラス、オフラインで利用可）
- モデル: PyTorch の小さな CNN（`Conv2d` / `MaxPool2d` / `ReLU` / `Linear`）
- 学習: `CrossEntropyLoss` + `Adam`
- 評価: 正解率・`confusion_matrix`・`classification_report`

このノートブックは学習コースのページ `practice-cnn-endtoend.html` と同じコードです。上から順に実行してください。CPU だけで数分で最後まで動きます。

In [ ]:
!pip install torch scikit-learn matplotlib

## Step 1. データ入手：手描き数字を読み込む

`load_digits` は 8x8 のグレースケール手描き数字を 1797 枚集めたデータセットです。各画素は 0..16 の整数、正解ラベルは 0..9。まず形と中身を確認します。

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.images   # (1797, 8, 8) 各画素は 0..16 の整数
y = digits.target   # (1797,)      正解ラベル 0..9

print("images:", X.shape, "labels:", y.shape, "classes:", np.unique(y))
print("pixel min/max:", X.min(), X.max())

試しに数枚を表示してみます。粗いながらも数字の形が見て取れます。

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 5, figsize=(8, 2))
for i in range(5):
    ax[i].imshow(X[i], cmap="gray_r")
    ax[i].set_title(f"label: {y[i]}")
    ax[i].axis("off")
plt.tight_layout()
plt.show()

## Step 2. 前処理：正規化・形状変換・分割

- **正規化**: 画素を 0..16 から 0..1 の実数へ（16で割る）。
- **形状変換**: CNN は `(枚数, チャネル, 高さ, 幅)` を受け取る。白黒なのでチャネルは1。`(1797, 8, 8)` を `(1797, 1, 8, 8)` に。
- **分割**: 学習用・検証用・テスト用の3つへ。`stratify` で各クラスの比率を保つ。テスト用は最後の評価まで触れない。

In [ ]:
import torch
from sklearn.model_selection import train_test_split

# 正規化（0..1）と形状変換（チャネル次元を追加）
X = (X / 16.0).astype("float32")
X = X[:, None, :, :]        # (N, 1, 8, 8)

# まずテストを切り分け、残りを学習用と検証用に分ける（stratifyで比率維持）
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=42, stratify=y_tr)

# PyTorch のテンソルに変換
Xtr = torch.tensor(X_tr); ytr = torch.tensor(y_tr)
Xva = torch.tensor(X_va); yva = torch.tensor(y_va)
Xte = torch.tensor(X_te); yte = torch.tensor(y_te)

print("train:", X_tr.shape[0], "val:", X_va.shape[0], "test:", X_te.shape[0])

## Step 3. CNN を定義する

畳み込み2層・プーリング2層・全結合2層の小さな CNN。プーリングで `8x8 -> 4x4 -> 2x2` と縮み、最後の全結合層の入力は `16*2*2 = 64`。活性化は `ReLU`、`padding=1` で端が縮まないようにします。

In [ ]:
import torch.nn as nn

torch.manual_seed(42)

class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),   # 8x8 -> 8x8
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 8x8 -> 4x4
            nn.Conv2d(8, 16, kernel_size=3, padding=1),  # 4x4 -> 4x4
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 4x4 -> 2x2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN()
n_params = sum(p.numel() for p in model.parameters())
print("params:", n_params)

## Step 4. 学習：損失を下げて重みを育てる

損失関数は `CrossEntropyLoss`、最適化は `Adam`。ミニバッチごとに「順伝播 → 損失 → 誤差逆伝播（`loss.backward()`）→ 重み更新（`optimizer.step()`）」をくり返し、各エポックの終わりに検証データで正解率を測ります。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 30
batch = 64
n = Xtr.shape[0]

for epoch in range(1, n_epochs + 1):
    model.train()
    perm = torch.randperm(n)              # 毎エポック順番をシャッフル
    total = 0.0
    for i in range(0, n, batch):
        idx = perm[i:i+batch]
        xb, yb = Xtr[idx], ytr[idx]
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()                   # 誤差逆伝播
        optimizer.step()                  # 重みを更新
        total += loss.item() * xb.size(0)

    # 検証データで正解率を測る（勾配は不要）
    model.eval()
    with torch.no_grad():
        va_pred = model(Xva).argmax(1)
        va_acc = (va_pred == yva).float().mean().item()
    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch {epoch:2d}  train_loss {total/n:.4f}  val_acc {va_acc:.3f}")

## Step 5. 評価：テストデータで実力を測る

これまで触れていないテストデータで最終評価。正解率に加え、`confusion_matrix` と `classification_report`（クラスごとの適合率・再現率・F値）で、どの数字を取り違えるかまで確認します。

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
with torch.no_grad():
    pred = model(Xte).argmax(1).numpy()

acc = (pred == y_te).mean()
print("test accuracy:", round(float(acc), 3))
print(confusion_matrix(y_te, pred))
print(classification_report(y_te, pred, digits=2))

誤分類したテスト例を数件取り出してみます。形の似た数字（8と1、9と4など）で誤りが出やすいことが分かります。

In [ ]:
# 誤分類したテスト例のインデックスを取り出す
wrong = np.where(pred != y_te)[0]
print("misclassified:", len(wrong), "of", len(y_te))
for i in wrong[:5]:
    print(f"index {i:3d}  true {y_te[i]}  pred {pred[i]}")

## まとめと発展

前処理・深層学習・評価を1本につなぎ、小さな CNN で手描き数字を9割前後の精度で分類できました。深層学習の実務は、データが大きくてもこの型（整える → 形を合わせる → 損失を下げる → 触っていないデータで評価）を踏みます。

### 発展課題
- **MNIST / Fashion-MNIST に広げる**（28x28）。`torchvision.datasets` から取得し、プーリング後の形と全結合層の入力サイズを合わせ直す。
- **ネットワークを深くする**：層やチャネルを増やして精度と学習時間の変化を見る。
- **過学習を抑える**：`nn.Dropout` やデータ拡張を加える。
- **学習曲線を描く**：エポックごとの損失と検証正解率を記録して `matplotlib` で可視化する。

参考書: 斎藤康毅『ゼロから作る Deep Learning』。